# 06 Redis 缓存

## 八股问答

### 1. 缓存雪崩、缓存穿透、缓存击穿的概念？

| 问题 | 发生条件 | 表现 |
| --- | --- | --- |
| 缓存雪崩 | 大量缓存同时过期，或 Redis 整体不可用 | 大量请求直接打到数据库，数据库压力暴涨 |
| 缓存穿透 | 查询的数据在缓存和数据库中都不存在 | 每次请求都穿过缓存访问数据库，恶意请求可打垮数据库 |
| 缓存击穿 | 某个热点 key 过期，大量请求同时访问它 | 单个高并发 key 失效，请求同时打到数据库 |

简单记忆：

- 雪崩：**一大批** key 同时失效。
- 穿透：**查不存在**的 key。
- 击穿：**单个热点 key** 失效。

### 2. 如何避免缓存雪崩、穿透、击穿？

**缓存雪崩：**

- 给 TTL 增加随机值，避免同一时间集中过期。
- 重要热点数据设置较长 TTL 或逻辑过期。
- 多级缓存：本地缓存 + Redis + 数据库。
- 数据库限流、熔断、降级，避免压垮 DB。
- 提高 Redis 高可用，避免单点故障。

**缓存穿透：**

- 参数校验，拒绝明显不存在的 key。
- 缓存空值并设置较短 TTL，例如 `nil`。
- 使用布隆过滤器/布谷鸟过滤器，先判断 key 是否存在。
- 限流和监控异常请求。

**缓存击穿：**

- 热点 key 不设置过期或使用逻辑过期。
- 请求加互斥锁，只有一个请求去数据库加载，其他请求等待或返回旧值。
- 本地缓存热点，减少 Redis 压力。
- 使用多副本或预热，降低单 key 失效影响。

### 3. 数据库和缓存如何保证一致性？

常用模式是 **Cache-Aside（旁路缓存）**：

```text
读：先查 Redis，命中直接返回；未命中查 MySQL，再写 Redis。
写：先更新 MySQL，再删除 Redis 中对应 key。
```

为什么更新数据库后删除缓存，而不是直接更新缓存：

- 更新缓存可能把旧数据写进去，且并发更新容易覆盖。
- 删除缓存可以延迟到下一次读时再加载。
- 但删除后、读请求未写入之前，可能读到旧值，存在短暂不一致窗口。

提高一致性：

- 设置合理的 TTL，保证最终一致。
- 删除失败时重试、异步删除或发送消息。
- 使用版本号、时间戳或数据库 binlog 同步。
- 对一致性要求极高的场景，先加锁或使用数据库事务外发消息。

Redis 不是数据库的替代品，必须把它当作可丢失、可重建的加速层。

### 4. 先更新数据库和先更新缓存的区别？

**先更新缓存，再更新数据库：**

- 如果数据库更新失败，缓存里已经是新值，缓存和数据库不一致。
- 并发场景下，旧数据库结果可能回写覆盖新缓存。
- 一般不建议。

**先更新数据库，再更新缓存：**

- 数据库更新失败时，可以避免继续写缓存。
- 但直接更新缓存在并发场景也可能被旧数据覆盖，因此比“更新”更稳的是“删除缓存”。

**先更新数据库，再删除缓存：**

- 下一次读会重新加载，缓存最终恢复正确。
- 并发读在删除前可能把旧值写回，期间仍可能存在短暂不一致。

比较：

| 顺序 | 风险 |
| --- | --- |
| 先缓存后数据库 | 数据库失败时缓存脏，风险最大 |
| 先数据库后更新缓存 | 并发回写可能覆盖新值 |
| 先数据库后删除缓存 | 大多数场景更稳，仍有短暂窗口 |

### 5. 讲一下延迟双删？

延迟双删用于缓解“先更新数据库、再删除缓存”时的并发窗口。

步骤：

1. 删除缓存。
2. 更新数据库。
3. 延时一小段时间。
4. 再次删除缓存。

```text
DEL cache
UPDATE database
sleep(delay)
DEL cache
```

为什么需要第二次删除：

- 第一次删除后，并发读请求可能把旧值重新写回缓存。
- 数据库更新完成后，再删一次，把这段窗口内的旧值清掉。

局限：

- 延时时长依赖业务经验，很难做到绝对准确。
- 第二次删除失败时仍可能不一致。
- 不能解决所有并发时序，尤其多个写请求交错时。
- 更可靠的方案是结合消息队列、binlog 监听、版本号或分布式锁。

面试时不要只背“删两次”，要说明这个延时是为了覆盖读回写窗口，并承认它不能保证强一致。


## 面试话术

“缓存穿透是查不存在的数据，用参数校验、空值缓存和布隆过滤器解决；缓存击穿是单个热点 key 失效，
用互斥锁、逻辑过期和热点预热解决；雪崩是大量 key 同时过期，用随机 TTL、多级缓存、限流和高可用解决。
数据库和缓存通常用 Cache-Aside，写时先更新数据库再删除缓存，并设置 TTL 兜底；
延迟双删是为了覆盖并发读回写旧值的窗口，但它只是经验性方案，不能保证强一致。”


## 自查清单

- [ ] 能区分雪崩、穿透、击穿及各自应对方案
- [ ] 能解释 Cache-Aside 的读写流程
- [ ] 能说明“先删缓存还是先更新缓存”的取舍
- [ ] 能讲清延迟双删的步骤和局限
